# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

c:\Users\Jason\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
pd_dir = os.getenv('PRICE_DATA')
pd_glob = glob(pd_dir+'/**/*.parquet', recursive=True)
df = dd.read_parquet(pd_glob)


In [4]:
df.columns

Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year'], dtype='object')

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.
df_shift = df.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(Adj_Close = x['Close'].shift(1))
)

df_shift = df.groupby('Ticker', group_keys=False).apply(
lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)

df_rets = df_shift.assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

df_hi_lo = df_rets.assign(
    hi_lo_range = lambda x: x['High'] - x['Low']
)

dd_feat = df_hi_lo

C:\Users\Jason\AppData\Local\Temp\ipykernel_36716\1607718028.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  df_shift = df.groupby('Ticker', group_keys=False).apply(
C:\Users\Jason\AppData\Local\Temp\ipykernel_36716\1607718028.py:6: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  df_shift = df.groupby('Ticker', group_keys=False).apply(


In [6]:
dd_feat

,Date,Ticker,Close,High,Low,Open,Volume,Year,Close_lag_1,returns,hi_lo_range
npartitions=3328,,,,,,,,,,,
,datetime64[ns],string,float64,float64,float64,float64,float64,int32,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [10]:
# Write your code below.

dd_feat = df_hi_lo.compute().reset_index().dropna()

dd_feat['returns_ma_10'] = dd_feat.groupby('Ticker')['returns'].transform(lambda x: x.rolling(10).mean())

dd_feat.info()


<class 'pandas.core.frame.DataFrame'>
Index: 357529 entries, 1 to 403455
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   index          357529 non-null  int64         
 1   Date           357529 non-null  datetime64[ns]
 2   Ticker         357529 non-null  string        
 3   Close          357529 non-null  float64       
 4   High           357529 non-null  float64       
 5   Low            357529 non-null  float64       
 6   Open           357529 non-null  float64       
 7   Volume         357529 non-null  float64       
 8   Year           357529 non-null  int32         
 9   Close_lag_1    357529 non-null  float64       
 10  returns        357529 non-null  float64       
 11  hi_lo_range    357529 non-null  float64       
 12  returns_ma_10  356953 non-null  float64       
dtypes: datetime64[ns](1), float64(9), int32(1), int64(1), string(1)
memory usage: 36.8 MB


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?  
No, it was not necessary to convert from dask dataframe to pandas dataframe since dask dataframes support operations done by pandas such as groupby and rolling.
+ Would it have been better to do it in Dask? Why?  
Yes, for large data sets it more memory efficient to use dask since it handles parallel oeprations, processes data in chunks, and easier to scale.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [X] Created a branch with the correct naming convention.
- [X] Ensured that the repository is public.
- [X] Reviewed the PR description guidelines and adhered to them.
- [X] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-5-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.